# Portfolio Returns & Risk vs a Benchmark

**Portfolio · Intermediate · ~15 min · pandas + yfinance · needs an internet connection**

Build a small multi-asset portfolio from real Yahoo Finance prices (through the `yfinance` package) and hold it up against the S&P 500 ETF, SPY. The notebook turns target weights into a daily portfolio return, grows $10,000 in both the portfolio and the benchmark, and measures what the portfolio paid you for the risk it took: return, volatility, Sharpe ratio, beta, maximum drawdown and tracking error. It needs an internet connection and writes nothing to disk.

**What you will see**
- The target weights and the latest prices of every holding, as tables
- A growth-of-$10,000 chart: portfolio vs benchmark
- A risk table for the portfolio, the benchmark and every holding
- The correlation matrix of daily returns
- Rolling 63-day (about three months) volatility for the portfolio and the benchmark

## 1. Parameters

`WEIGHTS` maps each Yahoo Finance ticker to its share of the portfolio. They should add up to 1 (100%); if they do not, the next cell rescales them and says so. The default mixes US large caps with a long-term Treasury bond ETF (TLT) and gold (GLD), so the effect of diversification is visible.

In [ ]:
# ── Parameters: edit these, then Run All (Ctrl+Shift+Enter) ──────────
WEIGHTS = {             # ticker -> target weight (should sum to 1.0)
    "AAPL": 0.20,
    "MSFT": 0.20,
    "JNJ":  0.15,
    "JPM":  0.15,
    "TLT":  0.15,       # 20+ year US Treasury bond ETF
    "GLD":  0.15,       # gold ETF
}
BENCHMARK = "SPY"       # S&P 500 ETF
PERIOD = "3y"           # "1y", "2y", "3y", "5y", "10y"
START_VALUE = 10_000    # dollars invested at the start
ROLL_WINDOW = 63        # rolling-volatility window in trading days (about 3 months)

## 2. Imports, chart style and the weights

The weights become a pandas Series indexed by ticker, which makes the portfolio return a single matrix product later on.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

# One consistent, quiet chart style for every figure in this notebook
plt.rcParams.update({
    "axes.prop_cycle": plt.cycler(color=["#2a78d6", "#eb6834", "#1baf7a", "#eda100",
                                         "#e87ba4", "#008300", "#6250d6", "#e34948"]),
    "axes.grid": True, "axes.grid.axis": "y", "grid.color": "#e1e0d9", "grid.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False, "axes.titleweight": "bold",
    "lines.linewidth": 1.6, "legend.frameon": False,
})

weights = pd.Series(WEIGHTS, dtype=float)
weights.index = weights.index.str.strip().str.upper()
if abs(weights.sum() - 1) > 1e-6:
    print(f"Weights add up to {weights.sum():.2f}, not 1 - rescaling them to 100%.")
    weights = weights / weights.sum()
weights.mul(100).round(1).rename("Weight (%)").to_frame()

## 3. Download prices

One `yf.download` call fetches split- and dividend-adjusted closes for every holding plus the benchmark; with several tickers the `Close` block has one column per ticker. The cell stops with a plain message if any ticker came back empty, then keeps only the days on which **every** ticker traded, so all the returns line up.

In [ ]:
tickers = list(dict.fromkeys(list(weights.index) + [BENCHMARK]))   # holdings + benchmark, no duplicates
data = yf.download(tickers, period=PERIOD, auto_adjust=True, progress=False)
if data.empty:
    raise ValueError(f"No price data for {tickers} - check the symbols or your connection")

closes = data["Close"]
missing = [t for t in tickers if t not in closes or closes[t].isna().all()]
if missing:
    raise ValueError(f"No price data for {missing} - check the symbol(s) or your connection")

closes = closes[tickers].dropna()      # common trading days only, columns in our order
print(f"{len(closes)} common trading days, {closes.index[0]:%Y-%m-%d} to {closes.index[-1]:%Y-%m-%d}")
closes.tail(5).round(2)

## 4. Daily portfolio returns

Each asset's daily return is the `pct_change()` of its close. The portfolio's return on a day is the weighted sum of its assets' returns: `asset_returns @ weights`.

**Assumption: the portfolio is rebalanced back to its target weights every day, at no cost.** That keeps the maths to one line. A real buy-and-hold portfolio lets the weights drift between rebalances, so its return differs a little; the *Portfolio Weights & Returns* template shows that drift.

In [ ]:
returns = closes.pct_change().dropna()
port = (returns[weights.index] @ weights).rename("Portfolio")   # weighted sum, rebalanced daily
bench = returns[BENCHMARK]

daily = pd.concat([port, bench], axis=1) * 100
print("Daily returns in percent, most recent first")
daily.iloc[::-1].head(10).round(2)

## 5. Growth of $10,000

Compounding the daily returns, `(1 + r).cumprod()`, gives the value of the investment over time. The gap between the lines at the right edge is the difference in total return; the shape of each line tells you how bumpy the ride was.

In [ ]:
growth = START_VALUE * (1 + pd.concat([port, bench], axis=1)).cumprod()

fig, ax = plt.subplots(figsize=(10, 4.5))
for name in growth.columns:
    ax.plot(growth[name], label=name)
    # label each line's final value at its right end
    ax.annotate(f"${growth[name].iloc[-1]:,.0f}", xy=(growth.index[-1], growth[name].iloc[-1]),
                xytext=(4, 0), textcoords="offset points", va="center", fontsize=9)
ax.axhline(START_VALUE, color="#898781", linewidth=0.8, linestyle="--")
ax.set_title(f"Growth of ${START_VALUE:,.0f}: portfolio vs {BENCHMARK}")
ax.set_xlabel("Date")
ax.set_ylabel("Value ($)")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"${v:,.0f}"))
ax.legend(loc="upper left")
plt.tight_layout()

## 6. Risk table

One row per return stream, all annualised with 252 trading days:

- **Annual return**: compound annual growth rate over the whole period.
- **Volatility**: standard deviation of daily returns × √252.
- **Sharpe**: annualised mean return ÷ volatility (risk-free rate set to 0).
- **Beta vs SPY**: covariance with the benchmark ÷ benchmark variance. 1.0 moves one-for-one with the market, 0.5 half as much.
- **Max drawdown**: the worst peak-to-trough fall of the growth curve.
- **Tracking error**: annualised volatility of the *difference* from the benchmark's return, i.e. how far you stray from the index. The benchmark's own row is 0 by definition.

In [ ]:
def risk_stats(r, bench, days=252):
    """Annualised return and risk numbers for one daily return series."""
    growth = (1 + r).cumprod()
    peak = growth.cummax().clip(lower=1)          # the start value counts as a peak
    return {
        "Annual return (%)": (growth.iloc[-1] ** (days / len(r)) - 1) * 100,
        "Volatility (%)": r.std() * np.sqrt(days) * 100,
        "Sharpe (rf = 0)": r.mean() / r.std() * np.sqrt(days),
        f"Beta vs {BENCHMARK}": r.cov(bench) / bench.var(),
        "Max drawdown (%)": (growth / peak - 1).min() * 100,
        "Tracking error (%)": (r - bench).std() * np.sqrt(days) * 100,
    }

series = {"Portfolio": port, BENCHMARK: bench}
series.update({t: returns[t] for t in weights.index if t != BENCHMARK})
risk = pd.DataFrame({name: risk_stats(r, bench) for name, r in series.items()}).T
risk.index.name = "Series"
risk.round(2)

## 7. Correlation matrix

Correlation runs from −1 (always move in opposite directions) to +1 (always move together). Diversification works through the low numbers: bonds (TLT) and gold (GLD) often correlate weakly with stocks, which is why the portfolio's volatility can sit below that of most of its holdings. The table includes the benchmark and the portfolio itself.

In [ ]:
corr = pd.concat([returns, port], axis=1).corr()
holdings = corr.loc[weights.index, weights.index].to_numpy()
avg_pair = holdings[np.triu_indices_from(holdings, k=1)].mean()   # average over distinct pairs
print(f"Average correlation between pairs of holdings: {avg_pair:.2f}")
corr.round(2)

## 8. Rolling volatility

Risk is not constant. Here the annualised volatility is recomputed every day over the trailing 63 trading days (about three months). Spikes line up with market stress; the gap between the lines shows how much the bonds and gold calm the portfolio down, or fail to when everything sells off together.

In [ ]:
rolling_vol = pd.concat([port, bench], axis=1).rolling(ROLL_WINDOW).std() * np.sqrt(252) * 100

fig, ax = plt.subplots(figsize=(10, 4))
for name in rolling_vol.columns:
    ax.plot(rolling_vol[name], label=name)
ax.set_title(f"Rolling {ROLL_WINDOW}-day annualised volatility")
ax.set_xlabel("Date")
ax.set_ylabel("Volatility (%)")
ax.legend(loc="upper left")
plt.tight_layout()

latest = rolling_vol.iloc[-1]
print(f"Latest {ROLL_WINDOW}-day volatility: portfolio {latest['Portfolio']:.1f}%, "
      f"{BENCHMARK} {latest[BENCHMARK]:.1f}%")

## Try next

- Replace `WEIGHTS` with your own holdings (4 to 10 liquid tickers work well) and run all cells again.
- Set `BENCHMARK = "QQQ"` for a tech-heavy portfolio, or `"ACWI"` for a global one.
- Drop TLT and GLD and watch the correlation, volatility and max drawdown rise.
- See **Mean-Variance Optimization** for choosing the weights instead of guessing them.

---
*— Fincept Notebook · part of Fincept Terminal. Edit any cell and press Ctrl+Enter to run.*